# 01 — Data exploration

Sanity-check the MIT-BIH (or synthetic-fallback) dataset and visualise per-class beat shapes
and per-client class distributions for the federated simulation.

Run all cells from the repo root.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from src.data.mitbih_loader import load_mitbih
from src.data.partitioner import partition
from src.data.preprocessing import stratified_train_test_split

In [ ]:
ds = load_mitbih(records=(100, 101, 103, 105), window_size=360)
print('Total beats:', len(ds), '  synthetic?', ds.is_synthetic)
print('Class distribution:', dict(zip(*np.unique(ds.y, return_counts=True))))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
for cls in range(5):
    idx = np.where(ds.y == cls)[0]
    if len(idx) == 0:
        axes[cls].set_title(f'class {cls} (none)')
        continue
    axes[cls].plot(ds.x[idx[0], 0], lw=0.8)
    axes[cls].set_title(f'class {cls} (n={len(idx)})')
fig.suptitle('Example beats per AAMI class')
plt.tight_layout()

In [ ]:
split = stratified_train_test_split(ds, test_fraction=0.2, seed=0)
parts = partition(split.train, num_clients=4, scheme='dirichlet', alpha=0.5, seed=0)
for i, p in enumerate(parts):
    counts = np.bincount(p.y, minlength=5)
    print(f'client {i}: total={len(p):4d}  per-class={counts.tolist()}')